# 0. Background

This is my implementation of the method from [Zhang et al 2004. *Estimation of saturated pixel values in digital color imaging*](zotero://select/items/1_HT3U4PF3). It is a Bayesian approach to estimating the true value of a saturated pixel, based on information in the unsaturated bands.

How the algorithm works:

**1. Estimate the prior distribution from clean water pixels.**

A multivariate normal prior is fit to a random sample of pixels that are: water, unsaturated in all bands, not edge-contaminated, and not invalid. Only water pixels are used so that the prior captures inter-band spectral relationships in the water column (rather than land surfaces, which are not used downstream)

**2. For each corrupt pixel, compute the conditional distribution.**

A pixel is *corrupt* if any band is saturated or edge-contaminated. Given the observed values in the clean bands, the conditional distribution of the corrupt bands is Zhang et al., Eq. 3.

**3. Estimate the true value from the conditional distribution.**

The treatment differs by corruption type:

- **Saturated bands**: the true value is known to exceed the saturation threshold $s$. The conditional mean is corrected upward using the truncated-normal mean (Zhang et al., Eq. 11):

- **Edge-contaminated bands**: the observed value is below threshold but biased by adjacency to saturated clusters. The conditional mean is used directly, with no truncation correction.

Pixels with more corrupt bands than `MAX_FILLED_BANDS` are set to NaN. The correction is applied image-wide (not water-only) because the pre-correction water mask misclassifies some saturated shallow water pixels as land; downstream water masking is applied after correction.

NOTE: Now that the algorithm accepts edge-contaminated pixels → need to rename MAX_N_OF_SATURATED bands to something more suitable (i.e `MAX_CORRUPT_BANDS`)

In [ ]:
import rasterio
from rasterio.windows import Window
from rasterio.enums import Resampling
import numpy as np
import rioxarray as rxr
from sklearn.model_selection import train_test_split
import pandas as pd

from joblib import Parallel, delayed

from pathlib import Path
import pystac

from tqdm.notebook import tqdm

from scipy import stats


In [ ]:
import sys
sys.path.insert(0, "..")

In [ ]:
from helpers.wv2 import load_mask
from helpers.shared import combine_masks, assert_grid_match

# 1. Config

In [ ]:
from config import *

# -- Input Data ------------------------------------------------------------------------------
STAC_DIR                    = Path("../stac/")
CATALOG                     = pystac.Catalog.from_file(STAC_DIR / "catalog.json")
INPUT_ASSET                 = "clip"
MASK_ASSETS                 = ["saturation-mask", "invalid-mask", "edge-mask", "water-mask"]

# -- Algorithm Parameters --------------------------------------------------------------------
MAX_FILLED_BANDS    = 3
NOISE_VARIANCE              = 1e-4      # See note below
SAMPLE_SIZE                 = 10_000
SEED                        = 42

# -- Processing Options ----------------------------------------------------------------------
CHUNKS                      = 512       # Chunk size to process everything in / for tiles
PARALLELISE                 = True      # Whether or not to use more than 1 CPU core
N_WORKERS                   = 8         # How many CPU cores to utilise
RAM_USAGE                   = 4096      # Max limit for loading into memory

# -- Output Options --------------------------------------------------------------------------
OUT_DIR                     = Path('../out/sat_corr/')
MKDIR                       = True      # Make OUT_DIR if it doesn't exist
OUT_NAME                    = "sat-corr"
OUT_EXT                     = "tif"

# -- Execution Flags -------------------------------------------------------------------------
OVERWRITE                   = True      # Whether the output should be overwritten

In [ ]:
OUT_DIR.mkdir(exist_ok=MKDIR)

# Note on Max Number of Saturated Bands (`MAX_FILLED_BANDS`)

`MAX_FILLED_BANDS` caps predictions at pixels with <=N saturated bands. The algorithm estimates saturated bands from non-saturated ones. Thus more saturated bands means fewer predictive bands and lower accuracy.

Zhang et al 2004 validated up to 2 saturated bands on RGB. With 8 multispectral bands, we can safely predict up to 3 saturated bands (leaving 5 for estimation). Most saturated pixels have <=2-3 bands, so `MAX_FILLED_BANDS`=2-3 is recommended.

# Note on Noise Variance (`NOISE_VARIANCE`)

According to the Zhang, $e_k$ is "a column vector with independent and identically distributed noise elements":

$$Y_k = X_k + e_k$$

Where $Y_k$ is the observed values of the nonsaturated sensors, i.e the true value $X_k$ plus measurement noise

The joint distribution of ${X_s}$ and ${Y_k}$ is:

$$\begin{pmatrix} 
X_s \\ 
X_k + e_k 
\end{pmatrix} = \begin{pmatrix} 
X_s \\ 
Y_k 
\end{pmatrix} \sim N\left(\begin{pmatrix} 
\mu_s \\ 
\mu_k 
\end{pmatrix}, \begin{pmatrix} 
S_s & S_{sk} \\ 
S_{sk} & S_k + S_{ek} 
\end{pmatrix}\right) \quad$$

where $S_{e_k}$ is an ${n_k} \times {n_k}$ diagonal matrix with identical diagonal elements $\sigma_e$ (i.e the standard deviation of error for the band)

## Specifying `NOISE_VARIANCE`

The paper provides no guidance on how to set the noise variance. Ideally, we would know the difference between observed and true sensor values to estimate the noise variance for each band directly. I was thinking that we might be able to use the signal-to-noise ratio (SNR) since a definition of SNR is signal/noise. Coffer et al. 2020 provide measurements of SNR for different bands of WorldView-2 over coastal seagrass systems, defined as "the ratio between the standard deviation and the mean of the satellite signal for each band" (Dadon et al. 2011). However, sadly they use the alt definition $SNR = \mu/\sigma$ (coefficient of variation), which is not directly translatable to Zhang. Thus I cannot uniquely determine​ Zhang's $\Sigma_{e_k}$ from Coffer's data alone without additional assumptions. This remains unresolved and is deferred for future investigation.

## Setting `NOISE_VARIANCE`

In the meantime, I assume that noise is:
1. negligible for a modern sensor
2. contributes minimally

Thus, for now, for an SR distribution normalised to [0,1], I set `NOISE_VARIANCE = 1e-4`.

In [ ]:
wv2_col = CATALOG.get_child("wv2-imagery")
wv2_items = list(wv2_col.get_items())

tqdm.write(f"{len(wv2_items)} item(s) found:")
for item in wv2_items:
    tqdm.write(f"ID: {item.id} | Date: {item.datetime} | Path: {item.assets[INPUT_ASSET].href}")

# Correction

## Prior Distribution

"If the image is not massively saturated, most of the pixel values will be valid. We can thus use the empirical mean and covariance of the nonsaturated pixels as the parameters of the prior multivariate normal distribution." I implement random sampling of all valid pixels (i.e with no saturation in any band)

In [ ]:
# def prior_dist(da, mask, sample_size=10000, seed=42):
#     n_bands, h, w = mask.shape
#     np.random.seed(seed)
    
#     n_pixels = h * w
#     if isinstance(sample_size, float) and 0 < sample_size < 1:
#         n_sample = int(n_pixels * sample_size)
#     else:
#         n_sample = int(sample_size)
#     n_sample = min(n_sample, n_pixels)
    
#     all_coords = np.arange(n_pixels)
#     sampled_coords = np.random.choice(all_coords, size=n_sample, replace=False)
#     rows = sampled_coords // w
#     cols = sampled_coords % w

#     with tqdm(total = 2, desc = "Estimating prior") as pbar:

#         da_sel = da.values[:, rows, cols]
#         pbar.update(1)

#         valid_all = ~np.any(mask[:, rows, cols], axis=0)
#         valid_px = da_sel[:, valid_all]
        
#         mu = np.mean(valid_px, axis=1)
#         sigma = np.cov(valid_px) if valid_px.shape[0] > 1 else np.eye(n_bands)
#         sigma += np.eye(n_bands) * 1e-6
#         pbar.update(1)
    
#     return mu, sigma

In [ ]:
class SatCorrector:
    """
    Bayesian saturation correction from Zhang et al 2004, implementing Eq. 11 for single band saturation.
    
    Extended to also correct edge-contaminated bands (pixels adjacent to saturated clusters).
    Saturated bands use the full truncated-normal estimator (Eq. 11); edge-contaminated bands
    use the conditional mean only (no truncation correction, since their value is below threshold).
    
    Note: Zhang & Brainard tested and validated the sequential procedure for up to 2 saturated bands
    (on RGB imagery). This implementation extends to multiple bands but may be less reliable beyond 2-3.
    """
    def __init__(self, prior_mean: np.ndarray, prior_cov: np.ndarray, noise_variance: float = 1e-4, max_filled_bands: int = 2):
        self.prior_mean = prior_mean
        self.prior_cov = prior_cov
        self.noise_variance = noise_variance
        self.max_filled_bands = max_filled_bands
        self.n_bands = len(prior_mean)
        
        if max_filled_bands < 1:
            raise ValueError("max_filled_bands must be at least 1")
        
        if max_filled_bands > self.n_bands:
            raise ValueError(f"max_filled_bands ({max_filled_bands}) cannot exceed n_bands ({self.n_bands})")

        try:
            self._cov_inverse = np.linalg.inv(prior_cov)
        except np.linalg.LinAlgError:
            self._cov_inverse = np.linalg.pinv(prior_cov)

    def _compute_Z(self, threshold, mu, sigma):
        """
        As per Eq 12 of Zhang et al 2004, compute Z = 1 - Phi((s - mu) / sqrt{sigma}). 
        """
        if sigma <= 0:
            return 0.0 if threshold > mu else 1.0
        std = np.sqrt(sigma)
        z_score = (threshold - mu) / std
        return 1.0 - stats.norm.cdf(z_score)

    def _compute_conditional_params(self, observed_k, corrupt_idx):
        """
        Compute conditional distribution parameters P(X_corrupt | Y_clean = k).
        
        From Equation 3 of the paper:
        mu_xs = mu_s + S_sk(S_k + S_ek)^(-1)(k - mu_k)
        S_xs = S_s - S_sk(S_k + S_ek)^(-1)S_sk^T
        """
        n_corrupt = len(corrupt_idx)
        clean_idx = np.setdiff1d(np.arange(self.n_bands), corrupt_idx)
        n_clean = len(clean_idx)

        if n_clean == 0:
            mu_cond = self.prior_mean[corrupt_idx]
            sigma_cond = self.prior_cov[np.ix_(corrupt_idx, corrupt_idx)]
            return mu_cond, sigma_cond
        
        mu_s = self.prior_mean[corrupt_idx]
        mu_k = self.prior_mean[clean_idx]
        sigma_s = self.prior_cov[np.ix_(corrupt_idx, corrupt_idx)]
        sigma_k = self.prior_cov[np.ix_(clean_idx, clean_idx)]
        sigma_sk = self.prior_cov[np.ix_(corrupt_idx, clean_idx)]

        innovation = observed_k - mu_k
        sigma_k_noisy = sigma_k + np.eye(n_clean) * self.noise_variance

        try:
            sigma_k_inv = np.linalg.inv(sigma_k_noisy)
        except np.linalg.LinAlgError:
            sigma_k_inv = np.linalg.pinv(sigma_k_noisy)

        mu_xs = mu_s + sigma_sk @ sigma_k_inv @ innovation
        sigma_xs = sigma_s - sigma_sk @ sigma_k_inv @ sigma_sk.T

        return mu_xs, sigma_xs

    def _estimate_single_saturated(self, mu, sigma, saturation_threshold=1.0):
        """
        Estimate true value for a single saturated band using Eq 11 of Zhang et al 2004.
        Uses truncated-normal correction because the observed value is known to exceed threshold.
        """
        if sigma <= 0:
            return mu

        std = np.sqrt(sigma)
        z_prob = self._compute_Z(saturation_threshold, mu, sigma)
        
        if z_prob < 1e-6:
            return mu
        
        z_score = (saturation_threshold - mu) / std
        exp_term = np.exp(-0.5 * z_score**2)
        numerator = std / np.sqrt(2 * np.pi) * exp_term
        correction = numerator / z_prob
        
        return mu + correction

    def correct_pixels(self, pixel, sat_mask, edge_mask=None, saturation_threshold=1.0):
        """
        Correct saturated and edge-contaminated values in a single pixel.
        - Saturated bands (sat_mask=True): estimated via truncated-normal (Eq. 11). Value is above threshold and the truncation correction applies.
        - Edge bands (edge_mask=True, sat_mask=False): estimated via conditional mean only. Value is below threshold but biased; no truncation correction.
        - Clean bands (neither saturated nor edge): used to condition the distribution.
        """
        corrected = pixel.copy().astype(float)
        sat_mask = sat_mask.astype(bool)
        edge_mask = edge_mask.astype(bool) if edge_mask is not None else np.zeros_like(sat_mask)

        # Bands to correct: saturated OR edge-contaminated
        corrupt_mask = sat_mask | edge_mask

        if not np.any(corrupt_mask):
            return corrected

        corrupt_idx = np.where(corrupt_mask)[0]
        clean_idx   = np.where(~corrupt_mask)[0]
        n_corrupt   = len(corrupt_idx)

        if n_corrupt > self.max_filled_bands:
            return np.full_like(corrected, np.nan)

        if len(clean_idx) == 0:
            return np.full_like(corrected, np.nan)

        observed_clean = pixel[clean_idx]
        mu_xs, sigma_xs = self._compute_conditional_params(observed_clean, corrupt_idx)

        for i, band_idx in enumerate(corrupt_idx):
            mu_cond    = mu_xs if np.isscalar(mu_xs) else mu_xs[i]
            sigma_cond = sigma_xs if np.isscalar(sigma_xs) else sigma_xs[i, i]

            if sat_mask[band_idx]:
                # Saturated: use truncated-normal estimator (Eq. 11)
                corrected[band_idx] = self._estimate_single_saturated(mu_cond, sigma_cond, saturation_threshold)
            else:
                # Edge-contaminated: use conditional mean directly
                corrected[band_idx] = mu_cond

        corrected = np.clip(corrected, 0.0, 1.0)
        return corrected

    def correct_image(self, img, sat_mask, edge_mask=None, saturation_threshold=1.0, verbose=False):
        """
        Correct saturation and edge contamination in an image.
        """
        n_bands, h, w = img.shape
        corrected = img.astype(float)

        edge_mask = edge_mask if edge_mask is not None else np.zeros_like(sat_mask)
        corrupt_mask = sat_mask | edge_mask

        pixels_needing_correction = np.any(corrupt_mask, axis=0)
        n_to_correct = np.sum(pixels_needing_correction)

        if verbose:
            pct = 100.0 * n_to_correct / (h * w)
            tqdm.write(f"Processing {n_to_correct:,} / {h*w:,} pixels ({pct:.2f}% need correction)")
            n_exceed = np.sum(np.sum(corrupt_mask, axis=0) > self.max_filled_bands)
            if n_exceed > 0:
                tqdm.write(f"Warning: {n_exceed:,} pixels exceed max_filled_bands — will be masked NaN")

        img_flat     = img.reshape(n_bands, -1)
        sat_flat     = sat_mask.reshape(n_bands, -1)
        edge_flat    = edge_mask.reshape(n_bands, -1)
        corr_flat    = corrected.reshape(n_bands, -1)

        pix_idx = np.where(pixels_needing_correction.ravel())[0]

        for idx in pix_idx:
            corr_flat[:, idx] = self.correct_pixels(
                img_flat[:, idx],
                sat_flat[:, idx],
                edge_flat[:, idx],
                saturation_threshold,
            )

        if verbose:
            tqdm.write(f"Corrected {len(pix_idx):,} pixels")

        return corrected

    def apply_correction(self, da, sat_mask, chunks, edge_mask=None, max_chunks=None, parallel=False, n_workers=1, RAM_mb=2048):
        """
        Apply saturation and edge-contamination correction chunk by chunk. Chunks with neither saturation nor edge contamination are yielded unmodified.
        """
        h, w = da.shape[1], da.shape[2]
        _edge = edge_mask if edge_mask is not None else np.zeros_like(sat_mask)

        chunks_list = [(i, j) for i in range(0, h, chunks) for j in range(0, w, chunks)]
        if max_chunks:
            chunks_list = chunks_list[:max_chunks]
        
        chunk_size_bytes = chunks * chunks * 8 * 4  # 8 bands * 4 bytes (float32)
        batch_size = max(1, int(RAM_mb * 1e6 / chunk_size_bytes))
        total_batches = (len(chunks_list) + batch_size - 1) // batch_size
        
        tqdm.write(f"RAM limit: {RAM_mb} MB | Batch size: {batch_size} chunks | Total batches: {total_batches}")
        
        for batch_start in range(0, len(chunks_list), batch_size):
            batch_num = batch_start // batch_size + 1
            batch_end = min(batch_start + batch_size, len(chunks_list))
            batch_chunks = chunks_list[batch_start:batch_end]
            
            loaded_batch = []
            for i_start, j_start in tqdm(batch_chunks, desc=f"[{batch_num}/{total_batches}] Loading batch", unit="chunk"):
                i_end = min(i_start + chunks, h)
                j_end = min(j_start + chunks, w)
                
                sat_chunk  = sat_mask[:, i_start:i_end, j_start:j_end]
                edge_chunk = _edge[:, i_start:i_end, j_start:j_end]

                # Skip chunks with no corruption
                if not np.any(sat_chunk) and not np.any(edge_chunk):
                    img_chunk = da.isel(y=slice(i_start, i_end), x=slice(j_start, j_end)).compute().values
                    yield i_start, j_start, img_chunk
                    continue
        
                img_chunk = da.isel(y=slice(i_start, i_end), x=slice(j_start, j_end)).compute().values
                loaded_batch.append((i_start, j_start, img_chunk, sat_chunk, edge_chunk))
            
            if not parallel:
                for i_start, j_start, img_chunk, sat_chunk, edge_chunk in tqdm(loaded_batch, desc="Correcting", unit="chunk"):
                    corrected_chunk = self.correct_image(img_chunk, sat_chunk, edge_chunk, verbose=False)
                    yield i_start, j_start, corrected_chunk
            else:
                def correct_one_chunk(data):
                    i_start, j_start, img_chunk, sat_chunk, edge_chunk = data
                    return i_start, j_start, self.correct_image(img_chunk, sat_chunk, edge_chunk, verbose=False)
                
                results = Parallel(n_jobs=n_workers, backend='loky', verbose=0)(
                    delayed(correct_one_chunk)(chunk_data) 
                    for chunk_data in tqdm(loaded_batch, desc="Correcting", unit="chunk")
                )
                
                for result in tqdm(results, total=len(loaded_batch), desc="Writing", unit="chunk"):
                    yield result

In [ ]:
def write_correction(out_path, da, corrected_chunks, profile_options, chunks=512):

    n_bands, h, w = da.shape
    crs = da.rio.crs
    transform = da.rio.transform()
    
    profile = dict(
        height=h,
        width=w,
        count=n_bands,
        crs=crs,
        transform=transform,
        **profile_options,
    )

    with rasterio.open(out_path, "w", **profile) as dst:
        for i_start, j_start, corrected in corrected_chunks:
            i_end = min(i_start + chunks, h)
            j_end = min(j_start + chunks, w)
            window = Window(j_start, i_start, j_end - j_start, i_end - i_start)
            dst.write(corrected, window=window)

    with rasterio.open(out_path, "r+") as dst:
        dst.build_overviews([2, 4, 8, 16], Resampling.average)
        dst.update_tags(ns="rio_overview", resampling="average")

    tqdm.write(f"Saved: {out_path.name}")

In [ ]:
pbar = tqdm(enumerate(wv2_items, start=1), desc="Correcting", unit="scene", dynamic_ncols=True, total=len(wv2_items))

val = {}

for i, item in pbar:
    pbar.set_description(f"[{i}/{len(wv2_items)}] Correcting: {item.id}")

    out_path = OUT_DIR / f"{item.id}-{OUT_NAME}.{OUT_EXT}"

    asset = pystac.Asset(
        href=str(out_path.resolve()),
        media_type=pystac.MediaType.GEOTIFF,
        roles=["data"],
        title = "Multispectral Saturation Correction",
        description = "Corrected for saturated and edge-contaminated pixels using Zhang & Brainard 2004"
    )
    item.add_asset(OUT_NAME, asset)

    if out_path.exists() and not OVERWRITE:
        tqdm.write(f"Skipping (exists): {out_path.name}")
        continue

    assert_grid_match(*[item.assets[a].href for a in [INPUT_ASSET] + MASK_ASSETS])

    da = rxr.open_rasterio(item.assets[INPUT_ASSET].href, chunks=CHUNKS)
    n_bands, h, w = da.shape

    # Mask

    masks = {}
    for mask in MASK_ASSETS:
        masks[mask] = rxr.open_rasterio(item.assets[mask].href, chunks=CHUNKS)

    tqdm.write(f"\n[Bands, H, W]: {da.shape} | CRS: {da.rio.crs} | dtype: {da.dtype}")

    sat_mask = (masks["saturation-mask"] == 1).compute().values[:n_bands] # (5, H, W)
    invalid_mask = (masks["invalid-mask"] == 1).compute().values # (1, H, W)
    edge_mask = (masks["edge-mask"] == 1).compute().values[:n_bands] # (5, H, W)
    water_mask = (masks["water-mask"] == 1).compute().values # (1, H, W)

    n_sat  = np.sum(sat_mask.any(axis=0))
    n_edge = np.sum((~sat_mask.any(axis=0)) & edge_mask.any(axis=0))
    n_tot  = da.shape[1] * da.shape[2]

    if n_sat == 0 and n_edge == 0:
        tqdm.write(f"Skipping (no saturation or edge-contamination)")
        continue

    tqdm.write(f"Saturated pixels: {n_sat:,} ({100.*n_sat/n_tot:.2f}%)")
    tqdm.write(f"Edge contaminated pixels: {n_edge:,} ({100.*n_edge/n_tot:.2f}%)")

    # Prior mask to estimate prior from clean water pixels only- excluding land, saturated, invalid, and edge-contaminated.
    prior_mask = sat_mask | invalid_mask | edge_mask | ~water_mask

    # Train / test split
    clean_flat = np.where(~prior_mask.any(axis=0).ravel())[0]
    pool = np.random.default_rng(SEED).choice(
        clean_flat,
        size = min(SAMPLE_SIZE, len(clean_flat)),
        replace = False
    )

    train_idx, val_idx = train_test_split(pool, test_size = 0.3, random_state = SEED)
    rows_tr, cols_tr = train_idx // w, train_idx % w
    rows_val, cols_val = val_idx // w, val_idx % w

    img = da.compute().values
    train_px = img[:, rows_tr, cols_tr]
    val_px = img[:, rows_val, cols_val]

    prior_mean = np.mean(train_px, axis = 1)
    prior_cov = np.cov(train_px) + np.eye(n_bands) * 1e-6

    tqdm.write(f"Prior mean: {prior_mean} | Prior cov: {prior_cov}")

    val[item.id] = dict(
        val_px = val_px,
        prior_mean = prior_mean,
        prior_cov = prior_cov,
        band_names = [str(b) for b in da.band.values]
    )

    corrector = SatCorrector(
        prior_mean,
        prior_cov,
        noise_variance = NOISE_VARIANCE,
        max_filled_bands = MAX_FILLED_BANDS,
    )

    corrected = corrector.apply_correction(
        da = da,
        sat_mask = sat_mask,
        edge_mask = edge_mask,
        chunks = CHUNKS,
        parallel = PARALLELISE,
        n_workers = N_WORKERS,
        RAM_mb = RAM_USAGE,
    )

    write_correction(
        out_path = out_path,
        da = da,
        corrected_chunks = corrected,
        chunks = CHUNKS,
        profile_options = PROFILE_FLOAT32,
    )

    tqdm.write(f"Catalog updated for {item.id}")

CATALOG.normalize_hrefs(str(STAC_DIR))
CATALOG.save(catalog_type=pystac.CatalogType.SELF_CONTAINED)
tqdm.write("Complete")

# Validation

- Leave-one-band-out validation; cycling through and leaving one band out, predicting based on the Bayesian method, comparing pred and true values

In [ ]:
assert_grid_match(*[item.assets[OUT_NAME].href for item in wv2_items])

In [ ]:
def predict_conditional(corrector, pixel, corrupt_idx):
    corrupt_idx = np.asarray(corrupt_idx)
    clean_idx = np.setdiff1d(np.arange(corrector.n_bands), corrupt_idx)

    if len(clean_idx) == 0:
        return corrector.prior_mean[corrupt_idx]
    mu_xs, _ = corrector._compute_conditional_params(pixel[clean_idx], corrupt_idx)
    return np.atleast_1d(mu_xs)

def band_metrics(true, pred):
    err = pred - true
    r2 = 1 - np.sum(err**2) / np.sum((true - true.mean()) ** 2)
    return dict(rmse = float(np.sqrt(np.mean(err ** 2))), r2 = float(r2), bias = float(err.mean()))

In [ ]:
val_records = []
pbar_val = tqdm(wv2_items, desc = "Validating", unit = "scene")
for item in pbar_val:
    if item.id not in val:
        continue

    s = val[item.id]
    val_px = s["val_px"]
    band_names = s["band_names"]
    n_bands = len(band_names)

    corrector = SatCorrector(
        s["prior_mean"],
        s["prior_cov"],
        noise_variance = NOISE_VARIANCE,
        max_filled_bands = MAX_FILLED_BANDS
    )

    true_per_band = {b: [] for b in range(n_bands)}
    pred_per_band = {b: [] for b in range(n_bands)}

    for i in range(val_px.shape[1]):
        pixel = val_px[:, i]
        if not np.all(np.isfinite(pixel)):
            continue
        for b in range(n_bands):
            pred = predict_conditional(corrector, pixel, np.array([b]))
            true_per_band[b].append(pixel[b])
            pred_per_band[b].append(pred[0])
    
    tqdm.write(f"\n Validating: {item.id}")
    for b, bname in enumerate(band_names):
        t = np.array(true_per_band[b])
        p = np.array(pred_per_band[b])
        m = band_metrics(t, p)
        val_records.append(dict(scene = item.id, band = bname, **m))
        tqdm.write(f"{bname:>6}: RMSE = {m['rmse']:.4f} | R2 = {m['r2']:.4f} | bias = {m['bias']:+.4f}")

df_val = pd.DataFrame(val_records)
if not df_val.empty:
    df_val.groupby("band")[["rmse", "r2", "bias"]].mean().round(4)


Dropping > REdge anyway so poor saturation dont matter here